In [4]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [5]:
# Charger le nouveau dataset
recipe = pd.read_csv("recipe/recipes_filtered.csv")
print(recipe.columns)
print(recipe.shape)

Index(['id', 'n_ingredients', 'ingredients', 'n_steps', 'minutes', 'steps',
       'log_minutes', 'avg_words_per_step', 'log_n_ingredients'],
      dtype='object')
(221813, 9)


# Calcul du score d'effort

Théorie :

L'effort culinaire = Temps (`log_minutes`) × Complexité technique (`avg_words_per_step`) × Ressources nécessaires (`log_n_ingredients`)
- Temps : contrainte temporelle directe
- Complexité : effort cognitif et technique requis
- Ingrédients : effort de préparation et logistique

### 1. Normalisation des variables (0-100)

In [6]:
def normalize_variables(recipe_data):
    # Créer les variables normalisées
    scaler = MinMaxScaler(feature_range=(0, 100))
    
    recipe_data['minutes_score'] = scaler.fit_transform(
        recipe_data[['log_minutes']]
    ).flatten()
    
    recipe_data['ingredients_score'] = scaler.fit_transform(
        recipe_data[['log_n_ingredients']]
    ).flatten()
    
    recipe_data['instructions_score'] = scaler.fit_transform(
        recipe_data[['avg_words_per_step']]
    ).flatten()
    
    return recipe_data

### 2. Score composite avec pondération

In [7]:
def calculate_effort_score(recipe_data, weights=None):
    """
    Calcule un score d'effort culinaire composite
    
    Poids suggérés basés sur l'impact sur l'effort :
    - temps : 40% (facteur principal)
    - complexité instructions : 35% (effort cognitif)
    - ingrédients : 25% (effort logistique)
    """
    
    if weights is None:
        weights = {
            'minutes': 0.40,
            'instructions': 0.35,
            'ingredients': 0.25
        }
    
    # Calculer le score composite
    recipe_data['effort_score'] = (
        recipe_data['minutes_score'] * weights['minutes'] +
        recipe_data['instructions_score'] * weights['instructions'] +
        recipe_data['ingredients_score'] * weights['ingredients']
    )
    
    return recipe_data

### 3. Catégorisation du score d'effort

In [10]:
def categorize_effort_level(recipe_data):
    """Catégorise le niveau d'effort en 5 niveaux"""
    
    def assign_effort_category(score):
        if score < 10:
            return 'Très Facile'
        elif score < 20:
            return 'Facile'
        elif score < 50:
            return 'Modéré'
        elif score < 70:
            return 'Difficile'
        else:
            return 'Très Difficile'
    
    recipe_data['effort_category'] = recipe_data['effort_score'].apply(
        assign_effort_category
    )
    
    return recipe_data

### Ajouter les variables au dataset

In [12]:
# Appliquer les fonctions pour calculer le score d'effort
recipe = normalize_variables(recipe)
recipe = calculate_effort_score(recipe)
recipe = categorize_effort_level(recipe)

# Vérifier les nouvelles colonnes créées
print("Nouvelles colonnes créées:")
new_columns = ['minutes_score', 'ingredients_score', 'instructions_score', 'effort_score', 'effort_category']
for col in new_columns:
    if col in recipe.columns:
        print(f"✅ {col}")
    else:
        print(f"❌ {col} - manquante")

# Afficher les statistiques du score d'effort
print(f"\nStatistiques du score d'effort:")
print(f"  • Moyenne: {recipe['effort_score'].mean():.1f}")
print(f"  • Médiane: {recipe['effort_score'].median():.1f}")
print(f"  • Min-Max: {recipe['effort_score'].min():.1f} - {recipe['effort_score'].max():.1f}")

# Afficher la répartition des catégories d'effort
print(f"\nRépartition des catégories d'effort:")
print(recipe['effort_category'].value_counts().sort_index())

# Sauvegarder le dataset avec le score d'effort culinaire
recipe.to_csv("recipe/culinary_effort.csv", index=False)
print(f"Dimensions finales: {recipe.shape}")
print(f"Colonnes: {list(recipe.columns)}")

Nouvelles colonnes créées:
✅ minutes_score
✅ ingredients_score
✅ instructions_score
✅ effort_score
✅ effort_category

Statistiques du score d'effort:
  • Moyenne: 37.0
  • Médiane: 37.9
  • Min-Max: 1.2 - 80.4

Répartition des catégories d'effort:
effort_category
Difficile          11978
Facile              9509
Modéré            199662
Très Difficile         1
Très Facile          663
Name: count, dtype: int64
Dimensions finales: (221813, 14)
Colonnes: ['id', 'n_ingredients', 'ingredients', 'n_steps', 'minutes', 'steps', 'log_minutes', 'avg_words_per_step', 'log_n_ingredients', 'minutes_score', 'ingredients_score', 'instructions_score', 'effort_score', 'effort_category']
